In [1]:
DATA TO DOCUMENT OF LANGCHAIN

SyntaxError: invalid syntax (1716412179.py, line 1)

In [2]:
!pip install langchain langchain_community langchain_core PyPDF pymupdf sentence-transformers chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 43.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 135.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 116.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 128.9 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: opente

In [3]:
from langchain_community.document_loaders.text import TextLoader

/tmp/ipykernel_1293/3315532343.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [4]:
loader = TextLoader("/content/python.txt" , encoding = "utf-8")
document = loader.load()

In [5]:
type(document)

list

In [6]:
from langchain_community.document_loaders.pdf import PyPDFLoader

Data to document


In [10]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [11]:
def load_all_pdf():
  folder_path = "/content/data"
  total_doc = 0
  all_doc = []

  for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
      pdf_path = os.path.join(folder_path,filename)
      loader = PyPDFLoader(pdf_path)
      doc = loader.load()
      all_doc.extend(doc)
      total_doc +=1

  print("total_doc",total_doc)
  print("total_pages",len(all_doc))
  return all_doc




In [12]:
all_pdf = load_all_pdf()

total_doc 1
total_pages 8


chunking


In [13]:
#WITH A LANGCHAIN MODULE
!pip install langchain_text_splitters

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 109, in _run_wrapper
    status = _inner_run()
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 102, in _inner_run
    return self.run(options, args)
           ~~~~~~~~^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 106, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 541, in run
    conflicts = self._determine_conflicts(to_install)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 662, in _determine_conflicts
    return check_install_conflicts(to_install)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/operations/check.py", line 110, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
            

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_doc(documents,chunk_size=500,chunk_overlap = 50):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap = chunk_overlap
  )

  chunked_docs = text_splitter.split_documents(documents)
  return chunked_docs

In [15]:
chunks = split_doc(all_pdf)

In [16]:
len(chunks)

102

Embedding

In [17]:
from sentence_transformers import SentenceTransformer

In [18]:
class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initializes the EmbeddingManager with a specific SentenceTransformer model.
        """
        self.model = SentenceTransformer(model_name)
        print(f"Loaded model: {model_name}")

    def generate_embeddings(self, texts: list) -> list:
        """
        Generates embeddings for a list of texts/strings and prints their shape.
        """
        embeddings = self.model.encode(texts)
        print(f"Embeddings shape: {embeddings.shape}")
        return embeddings

In [20]:
embedder = EmbeddingManager()
# embeddings = embedder.generate_embeddings([chunk.page_content for chunk in chunks])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded model: all-MiniLM-L6-v2


In [21]:
import chromadb
import uuid

In [37]:
class VectorStoreManager:
    def __init__(self, persist_directory="/content/data/chroma", collection_name="pdf_document"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.client = None
        self.collection = None
        self.initializestore()

    def initializestore(self):
        # Ensure directory exists
        os.makedirs(self.persist_directory, exist_ok=True)
        # Create the client and collection
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embedding in RAG"}
        )
        print(f"Initialized the vector store with collection as {self.collection_name}")
        print(f"Docs in collection: {self.collection.count()}")

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())





In [38]:
vector_store = VectorStoreManager()

Initialized the vector store with collection as pdf_document
Docs in collection: 0


In [42]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedder.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Embeddings shape: (102, 384)
total documents added in vector store= 102
docs in collection: 102


In [43]:
emebedding

array([[-0.12697941,  0.12199998, -0.04369551, ..., -0.02905141,
        -0.00243682,  0.02378318],
       [-0.06866066,  0.0906517 , -0.0643727 , ..., -0.02008373,
         0.00180153,  0.03503837],
       [-0.07387979,  0.08953606, -0.0414755 , ...,  0.00642245,
        -0.00853852, -0.0372391 ],
       ...,
       [-0.08219635,  0.04056253, -0.06423716, ..., -0.02898403,
        -0.03512603,  0.04783568],
       [-0.09923287,  0.05361547, -0.13774318, ..., -0.02750396,
         0.02507559,  0.00569296],
       [-0.0311666 , -0.03511841, -0.05752947, ..., -0.00062241,
         0.02426287, -0.03703321]], dtype=float32)

In [44]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [47]:
rag_retriever = RAGRetriever(embedder, vector_store)

In [59]:
rag_retriever.retrieve("iot")

Embeddings shape: (1, 384)
retrieved 2 documents


[{'id': 'doc_4d46c64d-7a7d-4a29-98c5-529a1b07ac1b',
  'document': 'includes radio frequency identiﬁcation (RFIDs), sensors, desk-\ntop computers, mobile phones, and more. One signiﬁcant\napplication of IoT is in healthcare, where it has the potential to\nimprove patient’s quality of life by facilitating faster access to\nbetter treatments. However, securing personal information col-\nlected, stored, or transmitted by medical IoT devices remains\na signiﬁcant challenge in healthcare IoT systems, as depicted',
  'metadata': {'creationdate': '2024-01-25T11:59:26+05:30',
   'page': 0,
   'author': '',
   'keywords': '',
   'source': '/content/data/research.pdf',
   'subject': '',
   'doc_index': 8,
   'content_length': 438,
   'page_label': '6111',
   'producer': 'Acrobat Distiller 23.0 (Windows)',
   'moddate': '2026-05-08T14:18:39+05:30',
   'creator': 'LaTeX with hyperref package',
   'total_pages': 8,
   'title': ''},
  'distance': 0.6800459623336792,
  'similarity_score': 0.3199540376

In [58]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [60]:
!git config --global user.email "aftabraza.ark16@gmail.com"
!git config --global user.name "Aftab-Raza

/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
